# Retail Sales — Exploratory Data Analysis

This notebook is the step-by-step EDA companion to the Streamlit dashboard. It uses the supplied `eda_practice_retail_sales.csv` and the reusable functions in `data_processing.py`.

**Objectives**
- Understand the dataset's structure, variables and quality.
- Clean inconsistent labels and invalid numeric values without inventing observations.
- Engineer gross sales, discounts, net order value and date/customer segments.
- Explore time, product, category, customer, city, payment, return and delivery patterns.
- Examine distributions, outliers and relationships between numeric variables.

> Sales after returns is an estimate that treats an order marked `Returned = Yes` as fully returned. There are no partial-return quantities or refund amounts in this dataset. No unique-customer count is calculated because there is no customer ID.


## 1. Imports and project paths

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / 'data_processing.py').exists():
    PROJECT_DIR = PROJECT_DIR.parent
sys.path.insert(0, str(PROJECT_DIR))

from data_processing import load_data, clean_data, calculate_kpis, build_quality_report

DATA_PATH = PROJECT_DIR / 'eda_practice_retail_sales.csv'
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
print('Project directory:', PROJECT_DIR)
print('Dataset exists:', DATA_PATH.exists())

Project directory: /mnt/data/Retail_Sales_EDA_Project
Dataset exists: True


## 2. Load the original dataset

In [2]:
raw_df = load_data(DATA_PATH)
print(f'Shape: {raw_df.shape[0]:,} rows × {raw_df.shape[1]} columns')
display(raw_df.head())

Shape: 508 rows × 14 columns


,Order_ID,Order_Date,Customer_Age,Gender,City,Product,Category,Quantity,Unit_Price_INR,Discount_Percent,Payment_Method,Customer_Rating,Delivery_Days,Returned
0,ORD1001,2025-09-15,NaN,Other,Delhi,Laptop,Computers,4,"53,319.16",10.00,UPI,3.00,10.00,No
1,ORD1002,2025-05-16,42.00,Female,Delhi,Monitor,Computers,4,"9,768.68",20.00,Net Banking,4.00,7.00,No
2,ORD1003,2025-06-16,60.00,Male,Bhopal,Keyboard,Accessories,3,777.26,0.00,Net Banking,2.00,7.00,No
3,ORD1004,2025-09-25,50.00,Other,Pune,Mouse,Accessories,2,617.67,10.00,UPI,2.00,5.00,Yes
4,ORD1005,2025-08-12,35.00,Other,Kanpur,Keyboard,Accessories,1,982.37,20.00,Cash,5.00,4.00,No


## 3. Initial inspection

In [3]:
raw_df.info()
display(raw_df.describe(include='all').T)
display(pd.DataFrame({
    'dtype': raw_df.dtypes.astype(str),
    'missing_count': raw_df.isna().sum(),
    'missing_percent': (raw_df.isna().mean() * 100).round(2),
    'unique_values': raw_df.nunique(dropna=True),
}))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 508 entries, 0 to 507
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Order_ID          508 non-null    object 
 1   Order_Date        508 non-null    object 
 2   Customer_Age      474 non-null    float64
 3   Gender            488 non-null    object 
 4   City              479 non-null    object 
 5   Product           508 non-null    object 
 6   Category          508 non-null    object 
 7   Quantity          508 non-null    int64  
 8   Unit_Price_INR    508 non-null    float64
 9   Discount_Percent  482 non-null    float64
 10  Payment_Method    494 non-null    object 
 11  Customer_Rating   468 non-null    float64
 12  Delivery_Days     486 non-null    float64
 13  Returned          508 non-null    object 
dtypes: float64(5), int64(1), object(8)
memory usage: 55.7+ KB


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Order_ID,508,500,ORD1218,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Order_Date,508,248,2025-06-08,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Customer_Age,474.00,NaN,NaN,NaN,40.59,14.98,-1.00,29.00,39.50,53.00,120.00
Gender,488,6,Other,167,NaN,NaN,NaN,NaN,NaN,NaN,NaN
City,479,15,Pune,78,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Product,508,7,Keyboard,91,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Category,508,3,Accessories,228,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Quantity,508.00,NaN,NaN,NaN,3.04,1.89,-2.00,2.00,3.00,4.00,30.00
Unit_Price_INR,508.00,NaN,NaN,NaN,"16,750.26","48,130.19",1.00,963.07,"11,151.78","19,884.72","999,999.00"
Discount_Percent,482.00,NaN,NaN,NaN,9.18,7.20,0.00,5.00,10.00,15.00,20.00


,dtype,missing_count,missing_percent,unique_values
Order_ID,object,0,0.00,500
Order_Date,object,0,0.00,248
Customer_Age,float64,34,6.69,51
Gender,object,20,3.94,6
City,object,29,5.71,15
Product,object,0,0.00,7
Category,object,0,0.00,3
Quantity,int64,0,0.00,8
Unit_Price_INR,float64,0,0.00,499
Discount_Percent,float64,26,5.12,5


## 4. Duplicate and category inspection

In [4]:
print('Exact duplicate rows:', raw_df.duplicated().sum())
print('Repeated Order_ID rows:', raw_df['Order_ID'].duplicated().sum())
for column in ['Gender', 'City', 'Product', 'Category', 'Payment_Method', 'Returned']:
    print(f'\n{column}:')
    display(raw_df[column].value_counts(dropna=False).to_frame('count'))

Exact duplicate rows: 8
Repeated Order_ID rows: 8

Gender:


,count
Gender,
Other,167
Male,152
Female,151
NaN,20
fem ale,9
FEMALE,6
male,3



City:


,count
City,
Pune,78
Jaipur,75
Bhopal,69
Kanpur,66
Lucknow,58
Delhi,56
Mumbai,55
NaN,29
MUMBAI,4



Product:


,count
Product,
Keyboard,91
Smartphone,87
Monitor,83
Headphones,73
Mouse,64
Laptop,60
Tablet,50


... (2 outputs left out)


Payment_Method:


,count
Payment_Method,
Credit Card,104
Net Banking,98
UPI,97
Debit Card,94
Cash,86
NaN,14
upi,8
debit card,6
Credit card,1



Returned:


,count
Returned,
No,438
Yes,70


## 5. Cleaning and feature engineering

In [5]:
df, cleaning_report = clean_data(raw_df)
print('Rows before:', cleaning_report['original_rows'])
print('Rows after:', cleaning_report['cleaned_rows'])
print('Exact duplicates removed:', cleaning_report['rows_removed_exact_duplicates'])
print('Remaining repeated Order_ID rows:', cleaning_report['remaining_duplicate_order_ids'])
print('Invalid numeric values set to missing:')
display(pd.Series(cleaning_report['invalid_values_set_missing'], name='count').to_frame())
print('IQR outlier counts:')
display(pd.Series(cleaning_report['iqr_outlier_counts'], name='count').to_frame())
display(df.head())

Rows before: 508
Rows after: 500
Exact duplicates removed: 8
Remaining repeated Order_ID rows: 0
Invalid numeric values set to missing:


,count
Customer_Age,3
Quantity,4
Unit_Price_INR,0
Discount_Percent,0
Customer_Rating,4
Delivery_Days,0


IQR outlier counts:


,count
Quantity,1
Unit_Price_INR,44
Gross_Sales,29
Net_Order_Value,26
Delivery_Days,0


,Order_ID,Order_Date,Customer_Age,Gender,City,Product,Category,Quantity,Unit_Price_INR,Discount_Percent,Payment_Method,Customer_Rating,Delivery_Days,Returned,Order_Year,Order_Month,Month_Name,Order_Day,Year_Month,Customer_Age_Group,Gross_Sales,Discount_Amount,Net_Order_Value,Is_Returned,Estimated_Net_Sales_After_Returns,Quantity_Outlier_IQR,Unit_Price_INR_Outlier_IQR,Gross_Sales_Outlier_IQR,Net_Order_Value_Outlier_IQR,Delivery_Days_Outlier_IQR
0,ORD1001,2025-09-15,NaN,Other,Delhi,Laptop,Computers,4.00,"53,319.16",10.00,UPI,3.00,10.00,No,2025,9,September,Monday,2025-09,<NA>,"213,276.64","21,327.66","191,948.98",False,"191,948.98",False,True,True,True,False
1,ORD1002,2025-05-16,42.00,Female,Delhi,Monitor,Computers,4.00,"9,768.68",20.00,Net Banking,4.00,7.00,No,2025,5,May,Friday,2025-05,36–45,"39,074.72","7,814.94","31,259.78",False,"31,259.78",False,False,False,False,False
2,ORD1003,2025-06-16,60.00,Male,Bhopal,Keyboard,Accessories,3.00,777.26,0.00,Net Banking,2.00,7.00,No,2025,6,June,Monday,2025-06,56–65,"2,331.78",0.00,"2,331.78",False,"2,331.78",False,False,False,False,False
3,ORD1004,2025-09-25,50.00,Other,Pune,Mouse,Accessories,2.00,617.67,10.00,UPI,2.00,5.00,Yes,2025,9,September,Thursday,2025-09,46–55,"1,235.34",123.53,"1,111.81",True,0.00,False,False,False,False,False
4,ORD1005,2025-08-12,35.00,Other,Kanpur,Keyboard,Accessories,1.00,982.37,20.00,Cash,5.00,4.00,No,2025,8,August,Tuesday,2025-08,26–35,982.37,196.47,785.90,False,785.90,False,False,False,False,False


## 6. Missing-value profile after cleaning

In [6]:
quality_report = build_quality_report(df)
display(quality_report.sort_values('Missing_Percent', ascending=False))
ax = quality_report.sort_values('Missing_Percent', ascending=True).plot(
    x='Column', y='Missing_Percent', kind='barh', figsize=(10, 8), legend=False,
    title='Missing Values by Column (%)'
)
ax.set_xlabel('Missing values (%)')
plt.tight_layout()
plt.show()

,Column,Data_Type,Missing_Count,Missing_Percent,Unique_Values
11,Customer_Rating,float64,44,8.80,5
2,Customer_Age,float64,37,7.40,49
19,Customer_Age_Group,string,37,7.40,6
4,City,string,30,6.00,7
21,Discount_Amount,float64,29,5.80,361
22,Net_Order_Value,float64,29,5.80,471
9,Discount_Percent,float64,25,5.00,5
24,Estimated_Net_Sales_After_Returns,float64,24,4.80,408
3,Gender,string,20,4.00,3
12,Delivery_Days,float64,20,4.00,10


/tmp/ipykernel_808/1801446323.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. KPI summary

In [7]:
kpis = calculate_kpis(df)
kpi_table = pd.DataFrame([{'KPI': key, 'Value': value} for key, value in kpis.items()])
display(kpi_table)

,KPI,Value
0,Total Orders,500.00
1,Units Ordered,"1,525.00"
2,Gross Sales (INR),"23,820,705.84"
3,Discount Amount (INR),"2,082,041.92"
4,Net Order Value (INR),"20,241,662.10"
5,Estimated Sales After Returns (INR),"18,313,121.23"
6,Average Order Value (INR),"40,483.32"
7,Average Unit Price (INR),"16,780.89"
8,Returned Orders,69.00
9,Return Rate (%),13.80


## 8. Univariate analysis — numeric variables

In [8]:
numeric_cols = ['Customer_Age', 'Quantity', 'Unit_Price_INR', 'Discount_Percent',
                'Customer_Rating', 'Delivery_Days', 'Gross_Sales', 'Net_Order_Value']
for column in numeric_cols:
    if column in df:
        fig, ax = plt.subplots(figsize=(8, 4))
        sns.histplot(data=df, x=column, kde=True, ax=ax, color='steelblue')
        ax.set_title(f'Distribution of {column}')
        plt.tight_layout()
        plt.show()

/tmp/ipykernel_808/1486682907.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/tmp/ipykernel_808/1486682907.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/tmp/ipykernel_808/1486682907.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_808/1486682907.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_808/1486682907.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/tmp/ipykernel_808/1486682907.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_808/1486682907.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/tmp/ipykernel_808/1486682907.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Univariate analysis — categorical variables

In [9]:
categorical_cols = ['Gender', 'City', 'Product', 'Category', 'Payment_Method', 'Returned', 'Order_Day']
for column in categorical_cols:
    if column in df:
        counts = df[column].fillna('Missing').value_counts().head(15).sort_values()
        fig, ax = plt.subplots(figsize=(9, 4))
        ax.barh(counts.index.astype(str), counts.values)
        ax.set_title(f'{column} distribution (top 15)')
        ax.set_xlabel('Count')
        plt.tight_layout()
        plt.show()

/tmp/ipykernel_808/3817472404.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_808/3817472404.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_808/3817472404.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/tmp/ipykernel_808/3817472404.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_808/3817472404.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_808/3817472404.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_808/3817472404.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 10. Time-series analysis

In [10]:
dated = df.dropna(subset=['Order_Date']).copy()
monthly = dated.groupby(dated['Order_Date'].dt.to_period('M').astype(str))['Net_Order_Value'].sum().reset_index()
monthly.columns = ['Month', 'Net_Order_Value']
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(monthly['Month'], monthly['Net_Order_Value'], marker='o')
ax.set_title('Monthly Net Order Value')
ax.set_ylabel('Net order value (INR)')
ax.tick_params(axis='x', rotation=35)
plt.tight_layout()
plt.show()

orders_by_day = df['Order_Day'].value_counts().reindex(
    ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
)
display(orders_by_day.to_frame('Orders'))

/tmp/ipykernel_808/883861247.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,Orders
Order_Day,
Monday,67
Tuesday,73
Wednesday,60
Thursday,66
Friday,78
Saturday,81
Sunday,75


## 11. Product and category analysis

In [11]:
category_summary = df.groupby('Category', dropna=False).agg(
    Orders=('Order_ID', 'nunique'), Units=('Quantity', 'sum'),
    Net_Order_Value=('Net_Order_Value', 'sum'), Average_Rating=('Customer_Rating', 'mean'),
).sort_values('Net_Order_Value', ascending=False)
display(category_summary)

product_summary = df.groupby('Product', dropna=False).agg(
    Orders=('Order_ID', 'nunique'), Units=('Quantity', 'sum'),
    Net_Order_Value=('Net_Order_Value', 'sum'),
).sort_values('Net_Order_Value', ascending=False)
display(product_summary)

fig, ax = plt.subplots(figsize=(9, 5))
product_plot = product_summary.sort_values('Net_Order_Value')
ax.barh(product_plot.index.astype(str), product_plot['Net_Order_Value'])
ax.set_title('Net Order Value by Product')
ax.set_xlabel('INR')
plt.tight_layout()
plt.show()

,Orders,Units,Net_Order_Value,Average_Rating
Category,,,,
Computers,141,422.00,"12,440,931.19",3.11
Electronics,135,429.00,"7,219,876.43",2.93
Accessories,224,674.00,"580,854.48",2.89


,Orders,Units,Net_Order_Value
Product,,,
Laptop,59,169.00,"8,716,767.01"
Smartphone,86,267.00,"4,765,650.83"
Monitor,82,253.00,"3,724,164.17"
Tablet,49,162.00,"2,454,225.60"
Headphones,71,194.00,"296,520.40"
Keyboard,89,242.00,"184,681.84"
Mouse,64,238.00,"99,652.23"


/tmp/ipykernel_808/1893675556.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 12. Customer profile and ratings

In [12]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(data=df, x='Customer_Age', bins=20, kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Customer Age')
rating_counts = df['Customer_Rating'].value_counts().sort_index()
axes[1].bar(rating_counts.index.astype(str), rating_counts.values)
axes[1].set_title('Customer Rating')
plt.tight_layout()
plt.show()

display(df.groupby('Customer_Age_Group', dropna=False).agg(
    Orders=('Order_ID', 'nunique'), Average_Rating=('Customer_Rating', 'mean'),
    Net_Order_Value=('Net_Order_Value', 'sum')
))

/tmp/ipykernel_808/1086236111.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,Orders,Average_Rating,Net_Order_Value
Customer_Age_Group,,,
18–25,85,2.99,"4,694,364.94"
26–35,100,2.99,"3,844,405.56"
36–45,93,3.13,"3,182,875.81"
46–55,87,2.92,"3,316,199.52"
56–65,94,2.59,"3,933,873.12"
Under 18,4,3.67,"41,286.03"
<NA>,37,3.44,"1,228,657.13"


## 13. City and payment analysis

In [13]:
city_summary = df.groupby('City', dropna=False).agg(
    Orders=('Order_ID', 'nunique'), Net_Order_Value=('Net_Order_Value', 'sum'),
    Average_Delivery_Days=('Delivery_Days', 'mean')
).sort_values('Net_Order_Value', ascending=False)
display(city_summary)

display(df['Payment_Method'].value_counts(dropna=False).to_frame('Orders'))
payment_summary = df.groupby('Payment_Method', dropna=False)['Net_Order_Value'].sum().sort_values(ascending=False)
display(payment_summary.to_frame('Net_Order_Value'))

,Orders,Net_Order_Value,Average_Delivery_Days
City,,,
Jaipur,78,"3,945,818.09",5.67
Delhi,57,"3,865,035.82",5.26
Bhopal,71,"2,761,314.05",5.75
Kanpur,67,"2,689,600.54",5.50
Pune,80,"2,188,323.67",5.21
Mumbai,56,"2,141,645.42",5.46
Lucknow,61,"1,683,538.06",5.58
<NA>,30,"966,386.44",5.86


,Orders
Payment_Method,
UPI,103
Credit Card,102
Net Banking,98
Debit Card,97
Cash,86
<NA>,14


,Net_Order_Value
Payment_Method,
UPI,"4,836,366.32"
Credit Card,"4,451,464.33"
Cash,"3,364,720.92"
Debit Card,"3,109,789.59"
Net Banking,"3,011,840.24"
<NA>,"1,467,480.71"


## 14. Discounts, returns and delivery

In [14]:
discount_return = df.groupby('Returned', dropna=False).agg(
    Orders=('Order_ID', 'nunique'), Average_Discount=('Discount_Percent', 'mean'),
    Net_Order_Value=('Net_Order_Value', 'sum'), Average_Rating=('Customer_Rating', 'mean'),
    Average_Delivery_Days=('Delivery_Days', 'mean')
)
display(discount_return)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=df, x='Returned', y='Delivery_Days', ax=axes[0], color='steelblue')
axes[0].set_title('Delivery Days by Return Status')
sns.boxplot(data=df, x='Returned', y='Customer_Rating', ax=axes[1], color='steelblue')
axes[1].set_title('Customer Rating by Return Status')
plt.tight_layout()
plt.show()

,Orders,Average_Discount,Net_Order_Value,Average_Rating,Average_Delivery_Days
Returned,,,,,
No,431,8.98,"18,313,121.23",2.97,5.60
Yes,69,10.94,"1,928,540.87",2.93,4.98


/tmp/ipykernel_808/1431442821.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 15. Correlation analysis

In [15]:
numeric = df.select_dtypes(include=np.number).corr(numeric_only=True)
plt.figure(figsize=(11, 8))
sns.heatmap(numeric, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Matrix of Numeric Variables')
plt.tight_layout()
plt.show()

/tmp/ipykernel_808/1406736469.py:2: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  plt.figure(figsize=(11, 8))


/tmp/ipykernel_808/1406736469.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 16. Outlier review

In [16]:
outlier_summary = []
for column in ['Quantity', 'Unit_Price_INR', 'Gross_Sales', 'Net_Order_Value', 'Delivery_Days']:
    flag = f'{column}_Outlier_IQR'
    if flag in df:
        outlier_summary.append({
            'Column': column, 'Outlier_Count': int(df[flag].sum()),
            'Outlier_Percent': round(float(df[flag].mean() * 100), 2),
        })
display(pd.DataFrame(outlier_summary))

# Inspect high-price records instead of automatically deleting them.
flag = df['Unit_Price_INR_Outlier_IQR']
display(df.loc[flag, ['Order_ID', 'Product', 'Category', 'Quantity', 'Unit_Price_INR', 'Net_Order_Value']]
        .sort_values('Unit_Price_INR', ascending=False).head(20))

,Column,Outlier_Count,Outlier_Percent
0,Quantity,1,0.20
1,Unit_Price_INR,44,8.80
2,Gross_Sales,29,5.80
3,Net_Order_Value,26,5.20
4,Delivery_Days,0,0.00


,Order_ID,Product,Category,Quantity,Unit_Price_INR,Net_Order_Value
177,ORD1178,Laptop,Computers,1.00,"999,999.00","999,999.00"
233,ORD1234,Monitor,Computers,5.00,"250,000.00","1,062,500.00"
152,ORD1153,Laptop,Computers,2.00,"68,093.53","122,568.35"
129,ORD1130,Laptop,Computers,3.00,"67,631.64","192,750.17"
483,ORD1484,Laptop,Computers,5.00,"66,615.71","283,116.77"
324,ORD1325,Laptop,Computers,4.00,"65,157.45","234,566.82"
335,ORD1336,Laptop,Computers,1.00,"64,970.06","55,224.55"
92,ORD1093,Laptop,Computers,4.00,"64,848.45","233,454.42"
296,ORD1297,Laptop,Computers,2.00,"64,775.57","103,640.91"
393,ORD1394,Laptop,Computers,4.00,"64,278.05",NaN


## 17. Export cleaned data and reports

In [17]:
output_dir = PROJECT_DIR / 'output'
output_dir.mkdir(exist_ok=True)
df.to_csv(output_dir / 'retail_cleaned.csv', index=False, encoding='utf-8-sig')
kpi_table.to_csv(output_dir / 'kpi_summary.csv', index=False, encoding='utf-8-sig')
quality_report.to_csv(output_dir / 'data_quality_report.csv', index=False, encoding='utf-8-sig')
print('Saved reports to:', output_dir)

Saved reports to: /mnt/data/Retail_Sales_EDA_Project/output


## 18. Interpretation checklist

Use the charts and tables above to write evidence-based observations:

1. Which month has the highest net order value? Is it also the month with the most orders?
2. Which product and category contribute the most net order value?
3. Which cities have the highest order value and order volume?
4. Which payment method is most frequently used?
5. Is average rating or delivery time different for returned and non-returned orders?
6. Which variables contain influential outliers? Are they data errors or plausible retail observations?
7. How much gross value is discounted, and how does discounting vary by product/category?

Do not claim causation from correlations. Treat the dataset as a sample of recorded orders rather than a complete representation of a real retail business.
